In [10]:
import os
import pandas as pd
import numpy as np

import config

import psycopg2
from sqlalchemy import text
from sqlalchemy import create_engine
from datetime import datetime

In [2]:
df = pd.read_csv(os.path.join(config.DATA_DIR, config.DATA_FILE), low_memory=False)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174104 entries, 0 to 174103
Data columns (total 66 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Record ID             174104 non-null  int64  
 1   Incident Year         174104 non-null  int64  
 2   Incident Month        174104 non-null  int64  
 3   Incident Day          174104 non-null  int64  
 4   Operator ID           174104 non-null  object 
 5   Operator              174104 non-null  object 
 6   Aircraft              174104 non-null  object 
 7   Aircraft Type         133074 non-null  object 
 8   Aircraft Make         131051 non-null  object 
 9   Aircraft Model        122439 non-null  object 
 10  Aircraft Mass         127320 non-null  float64
 11  Engine Make           123434 non-null  float64
 12  Engine Model          121988 non-null  object 
 13  Engines               127342 non-null  float64
 14  Engine Type           127282 non-null  object 
 15  

In [4]:
# --- CREATE CONNECTION ---
engine = create_engine(
    f"postgresql+psycopg2://{config.DB_USER}:{config.DB_PASSWORD}@{config.DB_HOST}:{config.DB_PORT}/{config.DB_NAME}"
)

In [12]:
RAW_TABLE_NAME = "raw_wildlife_strike"
LOG_TABLE_NAME = "ingestion_wildlife_strike_log"

# create new batch log entry
batch_id = datetime.now().strftime("%Y%m%d%H%M%S")
print(batch_id)

with engine.connect() as conn:
    conn.execute(
        text(f"""
            INSERT INTO ingestion_wildlife_strike_log (
                batch_id,
                source_file_name,
                ingestion_timestamp,
                process_status
            )
            VALUES (
                :batch_id,
                :source_file_name,
                :ingestion_timestamp,
                :process_status
            )
        """),
        {
            "batch_id": batch_id,
            "source_file_name": "database.csv",
            "ingestion_timestamp": datetime.now(),
            "process_status": "PENDING"
        }
    )
    conn.commit()

20260320211827


In [13]:

# add batch_id column
df["batch_id"] = batch_id

# export to temporary csv file
temp_file_name = f"temp_{batch_id}.csv"
temp_csv_path = os.path.join(config.DATA_DIR, temp_file_name)
df.to_csv(temp_csv_path, index=False)


In [14]:

CSV_PATH = os.path.expanduser(temp_csv_path)

conn = psycopg2.connect(
    host=config.DB_HOST,
    port=config.DB_PORT,
    dbname=config.DB_NAME,
    user=config.DB_USER,
    password=config.DB_PASSWORD
)
conn.autocommit = False

try:
    cur = conn.cursor()

    with open(CSV_PATH, "r", encoding="utf-8") as f:
        cur.copy_expert(
            """
            COPY raw_wildlife_strike (
                "Record ID",
                "Incident Year",
                "Incident Month",
                "Incident Day",
                "Operator ID",
                "Operator",
                "Aircraft",
                "Aircraft Type",
                "Aircraft Make",
                "Aircraft Model",
                "Aircraft Mass",
                "Engine Make",
                "Engine Model",
                "Engines",
                "Engine Type",
                "Engine1 Position",
                "Engine2 Position",
                "Engine3 Position",
                "Engine4 Position",
                "Airport ID",
                "Airport",
                "State",
                "FAA Region",
                "Warning Issued",
                "Flight Phase",
                "Visibility",
                "Precipitation",
                "Height",
                "Speed",
                "Distance",
                "Species ID",
                "Species Name",
                "Species Quantity",
                "Flight Impact",
                "Fatalities",
                "Injuries",
                "Aircraft Damage",
                "Radome Strike",
                "Radome Damage",
                "Windshield Strike",
                "Windshield Damage",
                "Nose Strike",
                "Nose Damage",
                "Engine1 Strike",
                "Engine1 Damage",
                "Engine2 Strike",
                "Engine2 Damage",
                "Engine3 Strike",
                "Engine3 Damage",
                "Engine4 Strike",
                "Engine4 Damage",
                "Engine Ingested",
                "Propeller Strike",
                "Propeller Damage",
                "Wing or Rotor Strike",
                "Wing or Rotor Damage",
                "Fuselage Strike",
                "Fuselage Damage",
                "Landing Gear Strike",
                "Landing Gear Damage",
                "Tail Strike",
                "Tail Damage",
                "Lights Strike",
                "Lights Damage",
                "Other Strike",
                "Other Damage",
                "batch_id"
            )
            FROM STDIN
            WITH (
                FORMAT CSV,
                HEADER TRUE
            )
            """,
            f
        )

    conn.commit()
    print("CSV loaded successfully into raw_wildlife_strike")

except Exception as e:
    conn.rollback()
    print(f"Load failed: {e}")

finally:
    cur.close()
    conn.close()

CSV loaded successfully into raw_wildlife_strike
